# Best Calibration

In [1]:
import sys; sys.path.insert(0, '../'); from lib import *;
figure_features()

# Set options for general visualitation
OPT  = {
    "MICRO_SEC":   True,                # Time in microseconds (True/False)
    "NORM":        False,               # Runs can be displayed normalised (True/False)
    "ALIGN":       True,                # Aligns waveforms in peaktime (True/False)
    "LOGY":        False,               # Runs can be displayed in logy (True/False)
    "SHOW_AVE":    "",                  # If computed, vis will show average (AveWvf,AveWvfSPE,etc.)
    "SHOW_PARAM":  False,               # Print terminal information (True/False)
    "CHARGE_KEY":  "ChargeAveRange",    # Select charge info to be displayed. Default: "ChargeAveRange" (if computed)
    "PEAK_FINDER": False,               # Finds possible peaks in the window (True/False)
    "LEGEND":      True,                # Shows plot legend (True/False)
    "SHOW":        True
    }


You don't have latex installed. Changing default configuration to tex=False


In [37]:
INPUT_FILE = "MegaCell_LAr"; OV = 2 #To Select the run: OV=0 -> run=103; OV=1 -> run=113; OV=2 -> run=105; OV=3 -> run=111  
PRESET ="ALL"
info = read_input_file(INPUT_FILE)  # Read input file
channels = [0,1,6,7]

#-------------------------------- LOAD RUNS ---------------------------------#
run_keys = ["CALIB_RUNS"]
nruns = dict.fromkeys(run_keys)
for key in run_keys:
    try:               nruns[key] = info[key][OV] # Store runs in dictionary
    except IndexError: nruns.pop(key)
print(nruns)

runs = dict.fromkeys(nruns.keys())
for run in runs: runs[run] = load_npy(np.asarray([nruns[run]]).astype(int),np.asarray(channels).astype(int),preset=PRESET,info=info,compressed=True)

{'CALIB_RUNS': 105}
load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!



In [38]:
RUN2PLOT='CALIB_RUNS'
AnaADC={};
Mean_AnaADC={};
Lim_ped={};
for c in channels:
    AnaADC[c]=(runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['PChannel']*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC'].T-runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPreTriggerMean'][0].T).T;
    Mean_AnaADC[c]=np.mean(AnaADC[c],axis=0);
    Lim_ped[c]=runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPedLim'];

# Limits
c=0; #Channel
signal=Mean_AnaADC[c][Lim_ped[c]:np.argmax(np.diff(Mean_AnaADC[c]))];
signal_invert=signal[::-1];
index=np.where(signal_invert[1:]>signal_invert[:-1])[0]+1;
start_charge=np.argmax(np.diff(Mean_AnaADC[c]))-index[0];
final=np.where(Mean_AnaADC[c][np.argmax(Mean_AnaADC[c]):]<np.mean(Mean_AnaADC[c][:start_charge]));

In [31]:
fig1=go.Figure()
fig1.add_trace(go.Scatter(x=np.arange(len(Mean_AnaADC[c])),y=Mean_AnaADC[c],name='Mean AnaADC',showlegend=True))
fig1.add_vline(x=start_charge-25)
fig1.add_vline(x=final[0][0]+np.argmax(Mean_AnaADC[c])+250)
fig1.update_layout(title='Integration range, run 105, channel 0',title_x=0.5)
fig1.show()

# Integration

In [17]:
from scipy import stats

c=0; #Select channel
charge=np.sum(AnaADC[c][:,start_charge:-200+final[0][0]+np.argmax(Mean_AnaADC[c])]*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['Sampling'],axis=1);
z_charge=np.abs(stats.zscore(charge))
charge_good=charge[np.where(z_charge<2)]

h, edges = np.histogram(charge_good,bins=500);
fig1=go.Figure()
fig1.add_trace(go.Bar(x=edges[:-1],y=h,name='Charge Histogram'))
fig1.update_yaxes(type='linear')
fig1.show()

# NEW RANGE OF INTEGRATION (Mejor Calib)

In [39]:
from scipy import stats

c=0; #Select channel
charge=np.sum(AnaADC[c][:,start_charge-25:final[0][0]+np.argmax(Mean_AnaADC[c])+10]*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['Sampling'],axis=1);
z_charge=np.abs(stats.zscore(charge))
charge_good=charge[np.where(z_charge<2)]

h, edges = np.histogram(charge_good,bins=500);
fig1=go.Figure()
fig1.add_trace(go.Bar(x=edges[:-1],y=h,name='Charge Histogram'))
fig1.update_yaxes(type='linear')
fig1.show()

In [53]:
from scipy.stats import norm

def gaussian(x, mu, sigma, amplitude):
    return amplitude * norm.pdf(x,mu,sigma)

def sum_of_gaussians(x, *params):
    n_gaussians = len(params) // 3
    result = np.zeros_like(x)
    for i in range(n_gaussians):
        mu, sigma, amplitude = params[i * 3], params[i * 3 + 1], params[i * 3 + 2]
        result += gaussian(x, mu, sigma, amplitude)
    return result

# Define an initial guess for the parameters (mean, standard deviation, and amplitude) of the Gaussian components
initial_guess = [0, 0.5e-6, 1, 11e-6, 0.5e-6, 0.5,21e-6,1e-6,0.01]

params, _ = curve_fit(sum_of_gaussians, edges[:-1], h/np.max(h), p0=initial_guess)

adjusted_distribution = sum_of_gaussians(edges, *params)

In [40]:
# Find peaks and valleys
from sklearn.cluster import KMeans

# Reshape data for KMeans input
data_reshaped = charge_good.reshape(-1, 1)

# Specify the number of peaks you want to find
num_peaks = 3

# Use KMeans to find peaks
kmeans = KMeans(n_clusters=num_peaks)
kmeans.fit(data_reshaped)

# Get the centroids (peak locations)
peaks = kmeans.cluster_centers_.flatten()

run=113;
fig1=go.Figure();
fig1.add_trace(go.Bar(x=edges[:-1],y=h/np.max(h),name='Charge Histogram',opacity=1))
fig1.add_trace(go.Scatter(x=peaks, y=np.zeros_like(peaks), mode='markers', marker=dict(color='red', size=10), name='Peaks'))
#fig1.add_trace(go.Scatter(x=edges,y=adjusted_distribution,name='Fitted distribution'))
fig1.update_yaxes(title='Normalized counts')
fig1.update_xaxes(title='Charge (ADC counts * s)')
fig1.update_layout(title='Channel = '+str(c)+' Run = '+str(run),title_x=0.5,bargap = 0,template='simple_white')
fig1.show()

In [41]:
print(peaks)

[1.44962051e-07 1.48194668e-05 6.61708197e-06]
